In [ ]:
# Colab Drive mount removed for GitHub — this notebook uses relative paths.
# Run it from notebooks/ with the repo checked out locally.
from pathlib import Path
PROJECT_DIR = Path("..").resolve()
print(PROJECT_DIR)


In [ ]:
import os

print(os.listdir(".."))

In [ ]:
import os

print(os.listdir(".."))

In [ ]:
from pathlib import Path

BASE = Path("..")
DATA = BASE / "data" / "sentence_labels.csv"
FASTTEXT_DIR = BASE / "results"  # FastText artifacts live in results/ now
RESULTS = BASE / "results"

FASTTEXT_DIR.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

In [ ]:
import pandas as pd
df = pd.read_csv(DATA)
df.head()
df["label"].value_counts()

In [ ]:
!pip install -q gensim scikit-learn pandas numpy

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import joblib
from gensim.models import FastText
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

BASE = Path("..")
DATA = BASE / "data" / "sentence_labels.csv"
FASTTEXT_DIR = BASE / "results"  # FastText artifacts live in results/ now
RESULTS = BASE / "results"
FASTTEXT_DIR.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

In [ ]:
def tokenize(text):
    return re.findall(r"[A-Za-z0-9]+(?:'[A-Za-z]+)?", text.lower())

def sentences_to_vectors(model, sentences, vector_size=64):
    vectors = []
    for sentence in sentences:
        words = tokenize(sentence)
        if not words:
            vectors.append(np.zeros(vector_size, dtype=np.float32))
            continue
        word_vecs = [model.wv[w] for w in words]
        vectors.append(np.mean(word_vecs, axis=0).astype(np.float32))
    return np.array(vectors)

df = pd.read_csv(DATA)
sentences = df["sentence"].tolist()
print("Sentences:", len(sentences))

tokenized = [tokenize(s) for s in sentences]
ft = FastText(vector_size=64, window=4, min_count=1, min_n=3, max_n=6, sg=1, workers=1, seed=7)
ft.build_vocab(corpus_iterable=tokenized)
ft.train(corpus_iterable=tokenized, total_examples=len(tokenized), epochs=50)

vectors = sentences_to_vectors(ft, sentences)
vector_df = pd.DataFrame(vectors, columns=[str(i) for i in range(64)])
result = pd.concat([df.reset_index(drop=True), vector_df], axis=1)

result.to_csv(FASTTEXT_DIR / "sentence_vectors.csv", index=False)
ft.save(str(FASTTEXT_DIR / "fasttext.model"))
print("Saved:", FASTTEXT_DIR / "sentence_vectors.csv")
print("Vector shape:", vectors.shape)

In [ ]:
df = pd.read_csv(FASTTEXT_DIR / "sentence_vectors.csv")
X = df.iloc[:, -64:]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_lr = scaler.fit_transform(X_train)
X_test_lr = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_lr, y_train)
lr_pred = lr.predict(X_test_lr)
print("Logistic Regression — accuracy:", accuracy_score(y_test, lr_pred), "F1:", f1_score(y_test, lr_pred))

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print("Random Forest — accuracy:", accuracy_score(y_test, rf_pred), "F1:", f1_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

joblib.dump(lr, RESULTS / "logistic_regression_model.pkl")
joblib.dump(rf, RESULTS / "random_forest_model.pkl")
joblib.dump(scaler, RESULTS / "scaler.pkl")
print("Models saved in results/")

In [ ]:
df = pd.read_csv(FASTTEXT_DIR / "sentence_vectors.csv")
rf = joblib.load(RESULTS / "random_forest_model.pkl")

X = df.iloc[:, -64:]
df["importance_score"] = rf.predict_proba(X)[:, 1]

TOP_K = 2  # sentences per article in the summary

rows = []
for article_id, grp in df.groupby("article_id"):
    ranked = grp.sort_values("importance_score", ascending=False)
    top = ranked.head(TOP_K).sort_values("sentence_id")
    rows.append({
        "article_id": article_id,
        "domain": grp["domain"].iloc[0],
        "summary": " ".join(top["sentence"]),
        "top_scores": top["importance_score"].tolist(),
    })

scores_out = RESULTS / "sentence_scores.csv"
summaries_out = RESULTS / "extractive_summaries.csv"

df[["article_id", "sentence_id", "sentence", "label", "importance_score"]].to_csv(scores_out, index=False)
pd.DataFrame(rows).to_csv(summaries_out, index=False)

print("Saved:", scores_out)
print("Saved:", summaries_out)
pd.DataFrame(rows)

In [ ]:
import pandas as pd

scores_path = "../results/sentence_scores.csv"

scores_df = pd.read_csv(scores_path)

print(scores_df.shape)
print(scores_df.columns.tolist())
display(scores_df.head(20))

In [ ]:
import pandas as pd

scores_path = "../results/sentence_scores.csv"

scores_df = pd.read_csv(scores_path)

print(scores_df.shape)
print(scores_df.columns.tolist())
display(scores_df.head(20))

In [ ]:
# Colab Drive mount removed for GitHub — this notebook uses relative paths.
# Run it from notebooks/ with the repo checked out locally.
from pathlib import Path
PROJECT_DIR = Path("..").resolve()
print(PROJECT_DIR)


In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path("..")
DATA = BASE / "data" / "sentence_labels.csv"

df = pd.read_csv(DATA)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nLabel counts:")
print(df["label"].value_counts())

print("\nArticles:")
print(df["article_id"].nunique())

print("\nSentences per article:")
print(df.groupby("article_id").size().value_counts())

In [ ]:
print("Missing values:")
print(df.isnull().sum())

In [ ]:
print("Domains:")
print(df["domain"].value_counts())

In [ ]:
print("Labels by domain:")
print(pd.crosstab(df["domain"], df["label"]))

In [ ]:
for article_id, group in df.groupby("article_id"):
    print("\nArticle:", article_id)
    display(group[["sentence_id", "sentence", "label"]].head(5))

    if article_id == df["article_id"].iloc[0]:
        break

In [ ]:
from sklearn.model_selection import train_test_split

# Get unique article IDs
article_ids = df["article_id"].unique()

# Split articles, NOT individual sentences
train_articles, test_articles = train_test_split(
    article_ids,
    test_size=0.2,
    random_state=42
)

# Create train/test dataframes
train_df = df[df["article_id"].isin(train_articles)].copy()
test_df = df[df["article_id"].isin(test_articles)].copy()

print("Training articles:", len(train_articles))
print("Testing articles:", len(test_articles))

print("Training sentences:", len(train_df))
print("Testing sentences:", len(test_df))

print("\nTraining labels:")
print(train_df["label"].value_counts())

print("\nTesting labels:")
print(test_df["label"].value_counts())

print("\nTest articles:")
print(sorted(test_articles))

In [ ]:
print(
    "Article overlap:",
    set(train_articles) & set(test_articles)
)

In [ ]:
!pip install -q gensim

In [ ]:
%run "../src/fasttext/fasttext_features.py"

In [ ]:
%run "../src/training/train_classifier.py"

In [ ]:
import pandas as pd

In [ ]:
%run "../src/summarization/summarize.py"

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../data/sentence_labels.csv"
)

print(df.shape)
print(df.columns.tolist())

print("\nFirst 10 rows:")
display(df.head(10))

print("\nLabel counts:")
print(df["label"].value_counts())

print("\nReference similarity:")
print(df["reference_similarity"].describe())

print("\nWeak supervision:")
print(df["weak_supervision"].value_counts())

In [ ]:
%run "../src/evaluation/evaluate_summarizer.py"

In [ ]:
%run "../src/fasttext/improved_features.py"

In [ ]:
%run "../src/training/train_improved_classifier.py"

In [ ]:
%run "../src/summarization/summarize_improved.py"

In [ ]:
%run "../src/evaluation/evaluate_improved.py"

In [ ]:
# %%
import sys

PROJECT_DIR = ".."

sys.path.insert(0, PROJECT_DIR)

from summarization.summarize_improved import summarize_article

print("Import successful!")

In [ ]:
# %%

from summarization.summarize_improved import summarize_article

article = """
PASTE A REAL ENGLISH ARTICLE HERE
"""

summary = summarize_article(
    article,
    top_k=5
)

print("SUMMARY:")
print(summary)

In [ ]:
# %%

article = """
Artificial intelligence is transforming many industries around the world.
Machine learning allows computers to learn patterns from large amounts of data.
Natural language processing enables computers to understand and process human language.
Researchers are developing new approaches to automatically summarize long documents.
Extractive summarization selects the most important sentences from an original article.
These systems can help users understand large amounts of information more efficiently.
Deep learning has significantly improved the performance of modern language applications.
Large datasets are commonly used to train machine learning systems.
Text classification is one of the important applications of natural language processing.
Search engines also use language models to improve the relevance of their results.
Automatic summarization can reduce the time required to read lengthy documents.
Researchers continue to investigate better methods for identifying important sentences.
Sentence position can provide useful information for determining sentence importance.
The first sentences of an article often contain important information about the topic.
However, important information can also appear near the end of an article.
Word count can provide another useful signal for sentence selection.
Longer sentences may contain more information than shorter sentences.
FastText can represent words and sentences using dense numerical vectors.
These vector representations can capture useful semantic and linguistic information.
Support Vector Machines can classify sentences based on their extracted features.
Combining semantic features with sentence-level features can improve classification.
The proposed system uses FastText vectors together with several additional features.
These features include sentence position and relative sentence position.
Word count and character count are also included in the feature representation.
Normalized word count provides information about sentence length relative to the article.
The machine learning model assigns an importance score to each sentence.
The highest-scoring sentences are selected to construct the extractive summary.
The selected sentences are then returned to their original order.
This approach preserves the structure and meaning of the original article.
The final system can provide a concise summary while retaining important information.
"""

summary = summarize_article(
    article,
    top_k=10
)

print("=" * 70)
print("30-SENTENCE ARTICLE")
print("=" * 70)

print("Number of sentences:",
      len(split_sentences(article)))

print("\n" + "=" * 70)
print("TOP 10 SUMMARY")
print("=" * 70)

print(summary)

In [ ]:
# %%

sentences = split_sentences(article)
scores = score_sentences(sentences)

ranked = sorted(
    enumerate(scores),
    key=lambda x: x[1],
    reverse=True
)

print("=" * 80)
print("SENTENCE RANKING")
print("=" * 80)

for rank, (index, score) in enumerate(ranked, start=1):
    print(
        f"{rank:2}. "
        f"S{index + 1:02} "
        f"Score={score:.4f} | "
        f"{sentences[index]}"
    )

In [ ]:
# %%

summary = summarize_article(
    article,
    top_k=5
)

print("=" * 70)
print("TOP 5 SUMMARY")
print("=" * 70)

print(summary)